# AITHOS Statistical Computations Validation

This notebook validates the computations for the AITHOS statistical analysis report. It uses the raw workbook as the authoritative data source and does not modify the raw file.

**Computed by:** Engr. Jamie Eduardo Rosal, MSCpE


## 1. Load Data, Define Constructs, and Validate Responses

The workbook contains Parts II-V only. Part I profile variables are unavailable and are therefore omitted from computation.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd

ROOT = Path(".").resolve()
DATA_FILE = ROOT / "AITHOS-SURVEY-RAW-DATA.xlsx"
ALPHA = 0.05

CONSTRUCTS = {
    "Expertise": {"sheet": "PART II.", "cols": [1, 2, 3, 4, 5]},
    "Trustworthiness": {"sheet": "PART II.", "cols": [7, 8, 9, 10, 11]},
    "Algorithmic Transparency": {"sheet": "PART III.", "cols": [1, 2, 3, 4, 5]},
    "Perceived Usefulness": {"sheet": "PART IV.", "cols": [1, 2, 3, 4, 5]},
    "Perceived Ease of Use": {"sheet": "PART IV.", "cols": [7, 8, 9, 10, 11]},
    "Quality of Content": {"sheet": "PART V.", "cols": [1, 2, 3, 4, 5]},
    "Message Consistency": {"sheet": "PART V.", "cols": [7, 8, 9, 10, 11]},
    "Audience Engagement": {"sheet": "PART V.", "cols": [14, 15, 16, 17, 18]},
}

def verbal_interpretation(mean):
    if mean >= 3.26:
        return "Strongly Agree / Very High"
    if mean >= 2.51:
        return "Agree / High"
    if mean >= 1.76:
        return "Disagree / Low"
    return "Strongly Disagree / Very Low"

def alpha_interpretation(alpha):
    if pd.isna(alpha):
        return "Not computable"
    if alpha >= 0.90:
        return "Excellent"
    if alpha >= 0.80:
        return "Good"
    if alpha >= 0.70:
        return "Acceptable"
    if alpha >= 0.60:
        return "Questionable"
    return "Low"

def correlation_interpretation(r):
    ar = abs(r)
    if ar >= 0.90:
        strength = "Very strong"
    elif ar >= 0.70:
        strength = "Strong"
    elif ar >= 0.50:
        strength = "Moderate"
    elif ar >= 0.30:
        strength = "Weak"
    elif ar >= 0.10:
        strength = "Very weak"
    else:
        strength = "Negligible"
    return f"{strength} {'positive' if r >= 0 else 'negative'}"

def p_value_from_r(r, n):
    if n <= 3 or not np.isfinite(r):
        return np.nan
    r = min(max(float(r), -0.999999999), 0.999999999)
    z = math.atanh(r) * math.sqrt(n - 3)
    return math.erfc(abs(z) / math.sqrt(2.0))

def cronbach_alpha(df):
    clean = df.dropna()
    k = clean.shape[1]
    if k < 2 or clean.shape[0] < 2:
        return np.nan
    item_variances = clean.var(axis=0, ddof=1)
    total_variance = clean.sum(axis=1).var(ddof=1)
    if total_variance == 0:
        return np.nan
    return (k / (k - 1)) * (1 - item_variances.sum() / total_variance)

raw = pd.read_excel(DATA_FILE, sheet_name=None, header=None)
construct_items = {}
respondent_ids = None

for name, spec in CONSTRUCTS.items():
    sheet = raw[spec["sheet"]]
    block = sheet.iloc[9:, [0] + spec["cols"]].copy()
    block.columns = ["Respondent"] + [f"{name} {i}" for i in range(1, 6)]
    block["Respondent"] = pd.to_numeric(block["Respondent"], errors="coerce").astype("Int64")
    for col in block.columns[1:]:
        block[col] = pd.to_numeric(block[col], errors="coerce")
    block = block.dropna(subset=["Respondent"]).set_index("Respondent").sort_index()
    construct_items[name] = block
    if respondent_ids is None:
        respondent_ids = block.index
    else:
        assert respondent_ids.equals(block.index), f"Respondent IDs do not align for {name}"

scored = pd.DataFrame(index=respondent_ids)
for name, block in construct_items.items():
    scored[name] = block.mean(axis=1)

scored["Overall Credibility"] = scored[["Expertise", "Trustworthiness"]].mean(axis=1)
scored["Overall Technology Acceptance"] = scored[["Perceived Usefulness", "Perceived Ease of Use"]].mean(axis=1)
scored["Overall Communication Strategy"] = scored[["Quality of Content", "Message Consistency", "Audience Engagement"]].mean(axis=1)

all_values = pd.concat(construct_items.values(), axis=1)
print("Respondents:", len(scored))
print("Respondent ID range:", int(scored.index.min()), "to", int(scored.index.max()))
print("Out-of-range Likert values:", int(((all_values < 1) | (all_values > 4)).sum().sum()))
print("Missing Likert values:", int(all_values.isna().sum().sum()))
for name, block in construct_items.items():
    print(f"{name}: {block.shape[1]} items x {block.shape[0]} respondents")

## 2. Descriptive Statistics by Construct

Weighted means are computed as arithmetic means of Likert responses.


In [ ]:
construct_order = [
    "Expertise", "Trustworthiness", "Overall Credibility",
    "Algorithmic Transparency",
    "Perceived Usefulness", "Perceived Ease of Use", "Overall Technology Acceptance",
    "Quality of Content", "Message Consistency", "Audience Engagement", "Overall Communication Strategy",
]

construct_summary = pd.DataFrame([
    {
        "Construct": name,
        "Mean": scored[name].mean(),
        "SD": scored[name].std(ddof=1),
        "Min": scored[name].min(),
        "Max": scored[name].max(),
        "Interpretation": verbal_interpretation(scored[name].mean()),
    }
    for name in construct_order
])
construct_summary

## 3. Item-Level Statistics

Each item is summarized by mean, standard deviation, and verbal interpretation.


In [ ]:
item_rows = []
for construct, block in construct_items.items():
    for idx, col in enumerate(block.columns, start=1):
        item_rows.append({
            "Construct": construct,
            "Item": idx,
            "Mean": block[col].mean(),
            "SD": block[col].std(ddof=1),
            "Interpretation": verbal_interpretation(block[col].mean()),
        })
item_summary = pd.DataFrame(item_rows)
item_summary

## 4. Reliability Analysis

Cronbach's alpha estimates internal consistency for each multi-item construct.


In [ ]:
reliability_rows = []
for construct, block in construct_items.items():
    alpha = cronbach_alpha(block)
    reliability_rows.append({
        "Scale": construct,
        "Items": block.shape[1],
        "Cronbach Alpha": alpha,
        "Interpretation": alpha_interpretation(alpha),
    })

combined = {
    "Overall AI Credibility": ["Expertise", "Trustworthiness"],
    "Overall Communication Strategy": ["Quality of Content", "Message Consistency", "Audience Engagement"],
    "Overall Technology Acceptance": ["Perceived Usefulness", "Perceived Ease of Use"],
}
for label, names in combined.items():
    block = pd.concat([construct_items[name] for name in names], axis=1)
    alpha = cronbach_alpha(block)
    reliability_rows.append({
        "Scale": label,
        "Items": block.shape[1],
        "Cronbach Alpha": alpha,
        "Interpretation": alpha_interpretation(alpha),
    })

reliability = pd.DataFrame(reliability_rows)
reliability

## 5. Pearson Correlation Analysis

Primary inferential analysis tests the relationship between AI credibility and communication strategy constructs.


In [ ]:
correlation_rows = []
predictors = ["Expertise", "Trustworthiness", "Overall Credibility"]
outcomes = ["Quality of Content", "Message Consistency", "Audience Engagement", "Overall Communication Strategy"]

for predictor in predictors:
    for outcome in outcomes:
        pair = scored[[predictor, outcome]].dropna()
        n = len(pair)
        r = pair[predictor].corr(pair[outcome])
        p = p_value_from_r(r, n)
        correlation_rows.append({
            "Predictor": predictor,
            "Outcome": outcome,
            "n": n,
            "r": r,
            "p-value": p,
            "Decision": "Significant" if p < ALPHA else "Not significant",
            "Interpretation": correlation_interpretation(r),
        })

correlations = pd.DataFrame(correlation_rows)
correlations

## 6. Publication-Style Visualizations

The following cell regenerates the same PNG figures inserted in the Word report.


In [ ]:
from build_aithos_analysis import generate_figures

figure_paths = generate_figures(scored, construct_summary, reliability, correlations)
for name, path in figure_paths.items():
    print(f"{name}: {path}")

try:
    from IPython.display import Image, display
    for path in figure_paths.values():
        display(Image(filename=str(path)))
except Exception:
    print("Figures generated. Inline display is available when this notebook is opened in Jupyter.")

![Construct mean profile](figures/construct_mean_profile.png)

![Credibility and communication strategy comparison](figures/credibility_strategy_comparison.png)

![Cronbach alpha reliability chart](figures/reliability_alpha_chart.png)

![Correlation heatmap](figures/correlation_heatmap.png)

![Overall credibility and communication strategy scatter plot](figures/credibility_strategy_scatter.png)


## 7. Optional Export of Computation Tables

This cell creates an Excel audit workbook from the computed tables.


In [ ]:
with pd.ExcelWriter("AITHOS_Computation_Tables.xlsx", engine="openpyxl") as writer:
    scored.reset_index().to_excel(writer, sheet_name="Respondent Scores", index=False)
    construct_summary.to_excel(writer, sheet_name="Construct Summary", index=False)
    item_summary.to_excel(writer, sheet_name="Item Summary", index=False)
    reliability.to_excel(writer, sheet_name="Reliability", index=False)
    correlations.to_excel(writer, sheet_name="Correlations", index=False)

print("Exported AITHOS_Computation_Tables.xlsx for audit/reference.")